# 8장 실습: Text2Cypher (POLE 데이터셋)

실습 가이드 8장 8-1 ~ 8-4의 코드를 순서대로 실행하는 노트북입니다. 설명은 실습 가이드를 함께 봅니다.

| 항목 | 내용 |
|---|---|
| 연결할 DB | POLE (샌드박스 또는 Docker) |
| .env 설정 | POLE DB의 NEO4J 네 줄 |

.env를 바꿨다면 노트북 상단의 `Restart`로 커널을 재시작한 뒤 처음부터 실행합니다. 연결 셀의 `connect_graph()`는 .env를 다시 읽지만, 앞 셀에서 만든 객체가 이전 DB를 가리킬 수 있으므로 처음부터 다시 실행합니다.

## 연결 확인

`Neo4jGraph`는 연결할 때 DB 스키마를 읽습니다. 이때 APOC 플러그인을 쓰므로, APOC가 없는 DB에서는 `Could not use APOC procedures` 오류가 납니다(Sandbox, AuraDB, 실습용 Docker 이미지에는 들어 있습니다).

출력된 레이블별 노드 수로 이 차시에 맞는 DB에 연결됐는지 확인합니다.

In [1]:
from tools.graphrag_tools import connect_graph

# .env의 접속 정보로 연결하고 레이블별 노드 수를 출력한다. expect의 레이블이 없으면 다른 DB에 연결됐다고 알려 준다.
# NEO4J는 접속 정보 묶음이다. 다른 LangChain 객체를 만들 때 **NEO4J로 그대로 넘긴다(실습 가이드 6-2).
# 연결 오류는 이 줄에서 난다. ServiceUnavailable: 서버가 꺼져 있거나 시작 중 / AuthError: 비밀번호 / APOC 오류: 플러그인 없음
graph, NEO4J = connect_graph(expect=["Crime", "Location", "PostCode"])

연결: bolt://localhost:7688 | DB: neo4j
{'label': 'Crime', 'n': 28762}
{'label': 'Location', 'n': 14904}
{'label': 'PostCode', 'n': 14196}
{'label': 'Vehicle', 'n': 1000}
{'label': 'Officer', 'n': 1000}
{'label': 'PhoneCall', 'n': 534}


## 1. 스키마 확인 (실습 가이드 8-1)

Text2Cypher 프롬프트에 넣을 스키마입니다. 데이터 구조가 바뀌었으면 `refresh_schema()`로 다시 읽습니다.

In [2]:
# 스키마를 다시 읽는다. 연결 이후 데이터 구조가 바뀌었을 수 있으므로 Text2Cypher 전에 최신으로 맞춘다.
graph.refresh_schema()
# 이 문자열이 LLM이 아는 DB 구조의 전부다. 여기 없는 레이블과 속성은 LLM이 알 수 없다.
print(graph.schema)

Node properties:
Person {surname: STRING, nhs_no: STRING, name: STRING, age: STRING}
Location {latitude: FLOAT, postcode: STRING, longitude: FLOAT, address: STRING}
Phone {phoneNo: STRING}
Email {email_address: STRING}
Officer {badge_no: STRING, rank: STRING, name: STRING, surname: STRING}
PostCode {code: STRING}
Area {areaCode: STRING}
PhoneCall {call_duration: STRING, call_time: STRING, call_date: STRING, call_type: STRING}
Crime {last_outcome: STRING, date: STRING, id: STRING, type: STRING}
Object {description: STRING, id: STRING, type: STRING}
Vehicle {model: STRING, reg: STRING, make: STRING, year: STRING}
Relationship properties:
FAMILY_REL {rel_type: STRING}
The relationships:
(:Person)-[:FAMILY_REL]->(:Person)
(:Person)-[:CURRENT_ADDRESS]->(:Location)
(:Person)-[:KNOWS]->(:Person)
(:Person)-[:HAS_PHONE]->(:Phone)
(:Person)-[:KNOWS_PHONE]->(:Person)
(:Person)-[:HAS_EMAIL]->(:Email)
(:Person)-[:KNOWS_SN]->(:Person)
(:Person)-[:KNOWS_LW]->(:Person)
(:Person)-[:PARTY_TO]->(:Crime)


## 2. OpenAI API로 직접 구현 (실습 가이드 8-3)

few-shot 예시는 user와 assistant 메시지 쌍으로 넣습니다. LLM을 두 번 호출합니다. 첫 번째는 쿼리를 만들고, 두 번째는 쿼리 결과를 문장으로 바꿉니다.

In [3]:
from openai import OpenAI

client = OpenAI()

# 시스템 프롬프트: 역할, 제약(스키마에 있는 것만), 출력 형식(쿼리만). {schema}는 호출할 때 채운다.
SYSTEM = """너는 Neo4j Cypher 전문가다. 아래 스키마에 있는 레이블, 관계, 속성만 사용해
사용자 질문에 답하는 Cypher 쿼리 하나를 작성한다. 설명 없이 쿼리만 출력한다.

스키마:
{schema}"""

# few-shot 예시: (질문, 정답 쿼리) 쌍. 관계 방향과 속성 이름을 예시로 보여 주면 정확도가 오른다(과제 2에서 비교).
FEW_SHOTS = [
    ("Piccadilly에서 발생한 범죄 건수는?",
     "MATCH (c:Crime)-[:OCCURRED_AT]->(l:Location {address: 'Piccadilly'})\nRETURN count(c) AS crime_count"),
    ("M22 8ER 우편번호 지역의 범죄 유형 분포는?",
     "MATCH (pc:PostCode {code: 'M22 8ER'})<-[:HAS_POSTCODE]-(l:Location)<-[:OCCURRED_AT]-(c:Crime)\nRETURN c.type, count(*) AS count ORDER BY count DESC"),
]

def text_to_cypher(question: str) -> str:
    messages = [{"role": "system", "content": SYSTEM.format(schema=graph.schema)}]
    # 예시를 user/assistant 대화로 넣어 '이런 질문에는 이렇게 답했다'는 형식을 보여 준다.
    for q, cypher in FEW_SHOTS:
        messages += [{"role": "user", "content": q}, {"role": "assistant", "content": cypher}]
    messages.append({"role": "user", "content": question})

    resp = client.chat.completions.create(model="gpt-4o", temperature=0, messages=messages)
    cypher = resp.choices[0].message.content.strip()
    # 쿼리만 출력하라고 해도 ```cypher 코드 블록으로 감싸 답할 때가 있어 벗겨 낸다.
    return cypher.removeprefix("```cypher").removeprefix("```").removesuffix("```").strip()

# 질문 → Cypher 생성 → DB 실행 → 결과를 근거로 답변 생성. GraphCypherQAChain이 하는 일을 직접 구현한 것이다.
def answer(question: str) -> str:
    cypher = text_to_cypher(question)
    rows = graph.query(cypher)
    print("생성된 Cypher:\n", cypher)
    resp = client.chat.completions.create(
        model="gpt-4o", temperature=0,
        messages=[
            {"role": "system", "content": "쿼리 결과만 근거로 한국어로 답한다."},
            # 결과가 많으면 토큰이 커지므로 20행까지만 넘긴다.
            {"role": "user", "content": f"질문: {question}\n쿼리 결과: {rows[:20]}"},
        ],
    )
    return resp.choices[0].message.content

print(answer("현재 수사 중인 범죄 사건의 개수는?"))

생성된 Cypher:
 MATCH (c:Crime {last_outcome: 'Under investigation'})
RETURN count(c) AS ongoing_investigations
현재 수사 중인 범죄 사건의 개수는 3,421건입니다.


## 3. GraphCypherQAChain으로 바꾸기 (실습 가이드 8-4)

프롬프트 템플릿 안에서 Cypher의 속성 중괄호는 `{{ }}`로 두 번 씁니다. 한 번만 쓰면 템플릿 변수로 해석돼 오류가 납니다.

In [4]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_neo4j import GraphCypherQAChain

# 2절의 SYSTEM과 FEW_SHOTS를 체인용 프롬프트 하나로 옮긴 것. {schema}와 {question}은 체인이 채운다.
# Cypher의 중괄호 {address: ...}는 템플릿 자리표시자와 겹치므로 {{ }}로 두 번 쓴다.
CYPHER_TEMPLATE = """너는 Neo4j Cypher 전문가다. 스키마에 있는 레이블, 관계, 속성만 사용해 쿼리 하나를 작성한다.
설명 없이 쿼리만 출력한다.

스키마:
{schema}

예시:
질문: Piccadilly에서 발생한 범죄 건수는?
쿼리: MATCH (c:Crime)-[:OCCURRED_AT]->(l:Location {{address: 'Piccadilly'}}) RETURN count(c) AS crime_count

질문: M22 8ER 우편번호 지역의 범죄 유형 분포는?
쿼리: MATCH (pc:PostCode {{code: 'M22 8ER'}})<-[:HAS_POSTCODE]-(l:Location)<-[:OCCURRED_AT]-(c:Crime) RETURN c.type, count(*) AS count ORDER BY count DESC

질문: {question}
쿼리:"""

cypher_prompt = PromptTemplate(input_variables=["schema", "question"], template=CYPHER_TEMPLATE)

# 2절의 text_to_cypher → graph.query → answer 세 단계를 체인 하나가 처리한다.
cypher_chain = GraphCypherQAChain.from_llm(
    llm=ChatOpenAI(model="gpt-4o", temperature=0),
    graph=graph,
    cypher_prompt=cypher_prompt,
    validate_cypher=True,             # 관계 방향 오류를 스키마 기준으로 교정
    return_intermediate_steps=True,   # 생성된 쿼리와 조회 결과를 함께 반환
    allow_dangerous_requests=True,    # LLM이 만든 쿼리를 실행하므로 명시적 동의가 필요
    verbose=True,
)

# 입력 키는 query, 최종 답변은 result, intermediate_steps[0]은 생성된 쿼리, [1]은 조회 결과다.
result = cypher_chain.invoke({"query": "범죄자는 아니지만 범죄자를 가장 많이 알고 있는 사람은?"})
print(result["result"])
print(result["intermediate_steps"][0]["query"])



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person)-[:KNOWS]->(c:Person)-[:PARTY_TO]->(:Crime)
WHERE NOT (p)-[:PARTY_TO]->(:Crime)
RETURN p.name, p.surname, count(c) AS known_criminals
ORDER BY known_criminals DESC
LIMIT 1

Full Context:
[{'p.name': 'Bonnie', 'p.surname': 'Gilbert', 'known_criminals': 8}]

> Finished chain.
Bonnie Gilbert는 범죄자는 아니지만 범죄자를 가장 많이 알고 있는 사람입니다.
MATCH (p:Person)-[:KNOWS]->(c:Person)-[:PARTY_TO]->(:Crime)
WHERE NOT (p)-[:PARTY_TO]->(:Crime)
RETURN p.name, p.surname, count(c) AS known_criminals
ORDER BY known_criminals DESC
LIMIT 1



## 실습 과제

위의 셀을 모두 실행한 뒤 진행합니다. 과제마다 문제, 정답 코드, 해설 순서로 되어 있습니다. 먼저 직접 작성해 본 뒤 정답 코드와 비교합니다.

### 과제 1. M22 8ER 지역의 최다 범죄 유형

"M22 8ER 지역에서 가장 많이 발생한 범죄 유형은?"을 2번(직접 구현)과 3번(체인) 방식으로 질문하고, 생성된 쿼리를 정답 쿼리와 비교해 검증하세요.

In [5]:
# 과제 1 정답
q = "M22 8ER 지역에서 가장 많이 발생한 범죄 유형은?"

print("[직접 구현]")
print(answer(q))

print("\n[GraphCypherQAChain]")
r = cypher_chain.invoke({"query": q})
print(r["result"])

# LLM 답변이 맞는지는 사람이 쓴 쿼리로 확인한다. Text2Cypher 결과는 항상 검증 대상이다.
print("\n[검증: 정답 쿼리로 전체 분포 확인]")
for row in graph.query("""
MATCH (:PostCode {code: 'M22 8ER'})<-[:HAS_POSTCODE]-(:Location)<-[:OCCURRED_AT]-(c:Crime)
RETURN c.type AS crime_type, count(*) AS cnt
ORDER BY cnt DESC
"""):
    print(row)

[직접 구현]
생성된 Cypher:
 MATCH (pc:PostCode {code: 'M22 8ER'})<-[:HAS_POSTCODE]-(l:Location)<-[:OCCURRED_AT]-(c:Crime)
RETURN c.type, count(*) AS count ORDER BY count DESC LIMIT 1
M22 8ER 지역에서 가장 많이 발생한 범죄 유형은 '재산 손괴 및 방화'입니다.

[GraphCypherQAChain]


> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (pc:PostCode {code: 'M22 8ER'})<-[:HAS_POSTCODE]-(l:Location)<-[:OCCURRED_AT]-(c:Crime) 
RETURN c.type 
ORDER BY count(*) DESC 
LIMIT 1



CypherSyntaxError: {neo4j_code: Neo.ClientError.Statement.SyntaxError} {message: Cannot use aggregation in ORDER BY if there are no aggregate expressions in the preceding RETURN (line 2, column 1 (offset: 95))
"RETURN c.type"
 ^} {gql_status: 50N42} {gql_status_description: error: general processing exception - unexpected error. Unexpected error has occurred. See debug log for details.}

#### 과제 1 해설

실행 결과 (2026-11-01)

| 방식 | 생성된 쿼리의 끝부분 | 답변 |
|---|---|---|
| 직접 구현 | `ORDER BY count DESC LIMIT 1` | '재산 피해 및 방화'(Criminal damage and arson) |
| GraphCypherQAChain | `ORDER BY count DESC LIMIT 1` | Criminal damage and arson |
| 검증(정답 쿼리) | `LIMIT` 없음 | Criminal damage and arson 2건, Drugs 2건, Possession of weapons 1건 |

- 두 방식 모두 쿼리는 문법적으로 맞고 실행도 됐지만 답이 불완전합니다. 1위가 Criminal damage and arson과 Drugs 두 유형(각 2건)인데, "가장 많이"라는 표현을 보고 LLM이 `LIMIT 1`을 붙여 하나가 사라졌습니다.
- few-shot 예시의 쿼리에는 `LIMIT`이 없었습니다. 그래도 LLM은 질문의 표현에 맞춰 `LIMIT 1`을 더했습니다. 예시를 그대로 따라 쓰는 것만은 아니라는 뜻입니다.
- 동률을 놓치지 않으려면 질문을 "동률이면 모두"처럼 구체적으로 쓰거나, 동률 처리 패턴(최댓값을 구한 뒤 같은 값을 모두 반환)을 예시에 넣습니다.

### 과제 2. few-shot 예시를 뺐을 때

3번 셀의 `CYPHER_TEMPLATE`에서 예시를 모두 뺀 프롬프트로 체인을 만들고, 과제 1과 같은 질문을 실행해 생성된 쿼리와 결과를 비교하세요.

In [ ]:
# 과제 2 정답
# 예시 부분만 뺀 템플릿. 나머지 설정을 cypher_chain과 같게 두어야 차이가 few-shot 때문이라고 말할 수 있다.
NO_SHOT_TEMPLATE = """너는 Neo4j Cypher 전문가다. 스키마에 있는 레이블, 관계, 속성만 사용해 쿼리 하나를 작성한다.
설명 없이 쿼리만 출력한다.

스키마:
{schema}

질문: {question}
쿼리:"""

no_shot_chain = GraphCypherQAChain.from_llm(
    llm=ChatOpenAI(model="gpt-4o", temperature=0),
    graph=graph,
    cypher_prompt=PromptTemplate(input_variables=["schema", "question"], template=NO_SHOT_TEMPLATE),
    validate_cypher=True,
    return_intermediate_steps=True,
    allow_dangerous_requests=True,
)

# 과제 1의 질문 q를 두 체인에 넣고 생성된 쿼리를 나란히 본다.
for name, chain in [("few-shot 있음", cypher_chain), ("few-shot 없음", no_shot_chain)]:
    r = chain.invoke({"query": q})
    print(f"== {name}")
    print(r["intermediate_steps"][0]["query"])
    print("결과 행:", r["intermediate_steps"][1]["context"])
    print("답변:", r["result"])



> Entering new GraphCypherQAChain chain...


Generated Cypher:
MATCH (pc:PostCode {code: 'M22 8ER'})<-[:HAS_POSTCODE]-(l:Location)<-[:OCCURRED_AT]-(c:Crime)
RETURN c.type, count(*) AS count
ORDER BY count DESC
LIMIT 1

Full Context:
[{'c.type': 'Criminal damage and arson', 'count': 2}]



> Finished chain.
== few-shot 있음
MATCH (pc:PostCode {code: 'M22 8ER'})<-[:HAS_POSTCODE]-(l:Location)<-[:OCCURRED_AT]-(c:Crime)
RETURN c.type, count(*) AS count
ORDER BY count DESC
LIMIT 1

결과 행: [{'c.type': 'Criminal damage and arson', 'count': 2}]
답변: M22 8ER 지역에서 가장 많이 발생한 범죄 유형은 Criminal damage and arson입니다.


== few-shot 없음
MATCH (l:Location)-[:HAS_POSTCODE]->(p:PostCode {code: 'M22 8ER'})-[:POSTCODE_IN_AREA]->(:Area)<-[:LOCATION_IN_AREA]-(loc:Location)<-[:OCCURRED_AT]-(c:Crime)
RETURN c.type AS crimeType, COUNT(c) AS occurrences
ORDER BY occurrences DESC
LIMIT 1

결과 행: [{'crimeType': 'Violence and sexual offences', 'occurrences': 175}]
답변: 폭력 및 성범죄가 M22 8ER 지역에서 가장 많이 발생한 범죄 유형입니다. 발생 건수는 175건입니다.


#### 과제 2 해설

실행 결과 (2026-11-01)

| 프롬프트 | 생성된 쿼리의 경로 | 결과 |
|---|---|---|
| few-shot 있음 | `PostCode ← Location ← Crime` | Criminal damage and arson 2건 |
| few-shot 없음 | `Location → PostCode → Area ← Location ← Crime` | Violence and sexual offences 175건 |

- 예시가 없자 LLM은 우편번호에서 지역(Area)으로 올라갔다가 그 지역의 모든 Location으로 내려오는 경로를 만들었습니다. 그래서 M22 8ER 한 곳이 아니라 지역 전체의 범죄를 셌습니다.
- 175건이라는 숫자와 "폭력 및 성범죄"라는 답은 그럴듯해 보이고 쿼리도 오류 없이 실행됐지만, 질문에 대한 답이 아닙니다. 쿼리를 출력해 읽어 보지 않으면 알 수 없는 오류입니다.
- few-shot 예시는 이 데이터에서 "우편번호 지역의 범죄"를 어떤 경로로 세는지 알려 주는 역할을 합니다. 스키마만으로는 여러 경로 중 어느 것이 질문의 뜻인지 LLM이 알 수 없습니다.

### 과제 3. 직접 구현과 체인 비교

아래 질문 3개를 2번의 `text_to_cypher()`와 3번의 `cypher_chain`에 넣어 생성된 쿼리를 나란히 비교하세요.

1. 현재 수사 중인 범죄 사건의 개수는?
2. Piccadilly에서 발생한 범죄 건수는?
3. 범죄자는 아니지만 범죄자를 가장 많이 알고 있는 사람은?

In [ ]:
# 과제 3 정답
# 앞에서 쓴 질문 세 개. 같은 질문에 두 방식이 만든 쿼리와 결과를 비교한다.
questions = [
    "현재 수사 중인 범죄 사건의 개수는?",
    "Piccadilly에서 발생한 범죄 건수는?",
    "범죄자는 아니지만 범죄자를 가장 많이 알고 있는 사람은?",
]
cypher_chain.verbose = False   # 비교 출력을 짧게

for q in questions:
    print("##", q)
    # 직접 구현은 쿼리 생성까지만 하고 실행은 graph.query로 따로 한다.
    direct = text_to_cypher(q)
    print("[직접 구현]", direct.replace("\n", " "))
    print("  결과:", graph.query(direct)[:3])
    r = cypher_chain.invoke({"query": q})
    print("[체인]    ", r["intermediate_steps"][0]["query"].replace("\n", " "))
    print("  결과:", r["intermediate_steps"][1]["context"][:3])
    print()

## 현재 수사 중인 범죄 사건의 개수는?


[직접 구현] MATCH (c:Crime {last_outcome: 'Under investigation'}) RETURN count(c) AS ongoing_investigations_count
  결과: [{'ongoing_investigations_count': 3421}]


[체인]     MATCH (c:Crime {last_outcome: 'Under investigation'}) RETURN count(c) AS ongoing_investigations 
  결과: [{'ongoing_investigations': 3421}]

## Piccadilly에서 발생한 범죄 건수는?


[직접 구현] MATCH (c:Crime)-[:OCCURRED_AT]->(l:Location {address: 'Piccadilly'}) RETURN count(c) AS crime_count
  결과: [{'crime_count': 166}]


[체인]     MATCH (c:Crime)-[:OCCURRED_AT]->(l:Location {address: 'Piccadilly'}) RETURN count(c) AS crime_count
  결과: [{'crime_count': 166}]

## 범죄자는 아니지만 범죄자를 가장 많이 알고 있는 사람은?


[직접 구현] MATCH (p:Person)-[:KNOWS]->(c:Person)-[:PARTY_TO]->(:Crime) WHERE NOT (p)-[:PARTY_TO]->(:Crime) RETURN p.name, p.surname, count(c) AS known_criminals ORDER BY known_criminals DESC LIMIT 1
  결과: [{'p.name': 'Bonnie', 'p.surname': 'Gilbert', 'known_criminals': 8}]


[체인]     MATCH (p:Person)-[:KNOWS]->(c:Person)-[:PARTY_TO]->(:Crime) WHERE NOT (p)-[:PARTY_TO]->(:Crime) RETURN p.name, p.surname, count(c) AS known_criminals ORDER BY known_criminals DESC LIMIT 1 
  결과: [{'p.name': 'Bonnie', 'p.surname': 'Gilbert', 'known_criminals': 8}]



#### 과제 3 해설

실행 결과 (2026-11-01)

| 질문 | 직접 구현과 체인의 쿼리 | 결과 | 정답 여부 |
|---|---|---|---|
| 현재 수사 중인 범죄 사건의 개수는? | 같음 | 3,421건 | 맞음 |
| Piccadilly에서 발생한 범죄 건수는? | 같음 | 166건 | 맞음 |
| 범죄자는 아니지만 범죄자를 가장 많이 알고 있는 사람은? | 같음 (`count(c)`, `LIMIT 1`) | Bonnie Gilbert 8 | 틀림 |

- 두 방식의 쿼리가 세 질문 모두 똑같았습니다. 같은 모델(gpt-4o), temperature 0, 비슷한 예시를 쓰면 같은 쿼리가 나오기 쉽습니다.
- 그래서 같은 실수도 똑같이 합니다. 세 번째 질문의 정답은 `count(DISTINCT c)`로 센 Anne Freeman(7명)입니다. `count(c)`는 범죄 2건에 연루된 지인을 두 번 세서 Anne Freeman과 Bonnie Gilbert가 8로 같아지고, `LIMIT 1`이 그중 하나만 남겼습니다(실습 가이드 8-5).
- 두 방식이 같은 답을 냈다고 그 답이 맞다는 뜻은 아닙니다. 검증은 생성된 쿼리를 읽고 정답 쿼리와 비교해서 합니다.
- 직접 구현은 코드가 짧아 프롬프트와 흐름이 다 보이고, 체인은 `validate_cypher`(관계 방향 교정), 중간 결과 반환 같은 기능이 붙어 있습니다.